# T2.1 – Create DBRepo Schema via REST API

Creates the `station` and `daily_observation` tables in DBRepo for the
**Frost Day Prediction in Vienna** experiment and attaches citable metadata
(DataCite Metadata Schema 4.4) to the database.

**Source data:** GeoSphere Austria – Messstationen Tagesdaten v2  
**Station:** Wien-Hohe Warte | **Period:** 2000–2023 | **Licence:** CC0

## 0 – Dependencies

```bash
pip install requests
```

In [44]:
import os
import json
import requests
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

load_dotenv()  # reads .env file

True

## 1 – Configuration

Enter the credentials for the shared group account registered on the test instance.

In [45]:
DBREPO_BASE_URL = "https://test.dbrepo.tuwien.ac.at"
API_BASE        = f"{DBREPO_BASE_URL}/api/v1"

USERNAME = os.getenv("DBREPO_USER")
PASSWORD = os.getenv("DBREPO_PASS")

if not USERNAME or not PASSWORD:
    raise ValueError("DBREPO_USER and DBREPO_PASS must be set in .env")

auth = HTTPBasicAuth(USERNAME, PASSWORD)
print(f"API base: {API_BASE}")

API base: https://test.dbrepo.tuwien.ac.at/api/v1


## 2 – Helper utilities

In [46]:
def post(endpoint: str, payload: dict) -> dict:
    url  = f"{API_BASE}{endpoint}"
    resp = requests.post(url, json=payload, auth=auth, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"POST {url} → {resp.status_code}\n{resp.text[:600]}")
    return resp.json()


def get(endpoint: str):
    url  = f"{API_BASE}{endpoint}"
    resp = requests.get(url, auth=auth, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"GET {url} → {resp.status_code}\n{resp.text[:600]}")
    return resp.json()


def delete(endpoint: str) -> None:
    url  = f"{API_BASE}{endpoint}"
    resp = requests.delete(url, auth=auth, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"DELETE {url} → {resp.status_code}\n{resp.text[:600]}")

## 3 – Find the MariaDB container

DBRepo manages one or more database engine containers identified by a UUID.
We need that UUID when creating a database.

In [47]:
containers = get("/container")
print(json.dumps(containers, indent=2))

[
  {
    "id": "6cfb3b8e-1792-4e46-871a-f3d103527203",
    "hash": null,
    "name": "mariadb-galera:11.3.2",
    "image": {
      "id": "d79cb089-363c-488b-9717-649e44d8fcc5",
      "name": "mariadb",
      "version": "11.1.3",
      "default": false
    },
    "quota": null,
    "count": 33,
    "internal_name": "mariadb_11_3_2"
  }
]


In [48]:
# Use the first available container (adjust index if the instance has several)
container_id = containers[0]["id"]
print(f"Container id: {container_id}")

Container id: 6cfb3b8e-1792-4e46-871a-f3d103527203


## 4 – Create the database

In [49]:
DB_NAME = "frost_day_prediction_in_vienna"

In [50]:
db_result   = post("/database", {
    "name":             DB_NAME,
    "container_id":     container_id,
    "is_public":        True,
    "is_schema_public": True
})
database_id = db_result["id"]
print(f"Created database  id: {database_id}")

Created database  id: a16a980a-3489-492b-adcf-74c70a07248b


## 5 – Create table: `station`

In [51]:
# check database state before creating tables
import json
db_info = get(f"/database/{database_id}")
print(json.dumps(db_info, indent=2))

{
  "id": "a16a980a-3489-492b-adcf-74c70a07248b",
  "name": "frost_day_prediction_in_vienna",
  "description": null,
  "tables": [],
  "views": [],
  "container": {
    "id": "6cfb3b8e-1792-4e46-871a-f3d103527203",
    "name": "mariadb-galera:11.3.2",
    "image": {
      "id": "d79cb089-363c-488b-9717-649e44d8fcc5",
      "name": "mariadb",
      "version": "11.1.3",
      "operators": [
        {
          "id": "130c6782-f830-11f0-8413-0e74369233d4",
          "value": "=",
          "documentation": "https://mariadb.com/kb/en/assignment-operators-assignment-operator/",
          "display_name": "Equal operator"
        },
        {
          "id": "130c693b-f830-11f0-8413-0e74369233d4",
          "value": "<=>",
          "documentation": "https://mariadb.com/kb/en/null-safe-equal/",
          "display_name": "NULL-safe equal operator"
        },
        {
          "id": "130c6a49-f830-11f0-8413-0e74369233d4",
          "value": "<",
          "documentation": "https://mariadb.com

In [ ]:
station_payload = {
    "name": "station",
    "description": "Meteorological station metadata including location and elevation",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {
            "name": "station_id",
            "type": "int",
            "size": None,
            "d": None,
            "description": "Unique numeric station identifier",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": False,
            "concept_uri": None,
            "unit_uri": None
        },
        {
            "name": "station_code",
            "type": "varchar",
            "size": 20,
            "d": None,
            "description": "Alphanumeric station code assigned by GeoSphere Austria",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": False,
            "concept_uri": None,
            "unit_uri": None
        },
        {
            "name": "name",
            "type": "varchar",
            "size": 100,
            "d": None,
            "description": "Human-readable station name",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": False,
            "concept_uri": None,
            "unit_uri": None
        },
        {
            "name": "latitude",
            "type": "decimal",
            "size": 8,
            "d": 5,
            "description": "WGS-84 latitude in decimal degrees",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": True,
            "concept_uri": None,
            "unit_uri": None
        },
        {
            "name": "longitude",
            "type": "decimal",
            "size": 8,
            "d": 5,
            "description": "WGS-84 longitude in decimal degrees",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": True,
            "concept_uri": None,
            "unit_uri": None
        },
        {
            "name": "altitude_m",
            "type": "int",
            "size": None,
            "d": None,
            "description": "Station elevation above sea level in metres",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": True,
            "concept_uri": None,
            "unit_uri": None
        }
    ],
    "constraints": {
        "primary_key": ["station_id"],
        "uniques": [["station_code"]],
        "checks": [],
        "foreign_keys": []
    }
}
station_result   = post(f"/database/{database_id}/table", station_payload)
station_table_id = station_result["id"]
print(f"Created table 'station'           id: {station_table_id}")

## 6 – Create table: `daily_observation`

In [ ]:
obs_payload = {
    "name": "daily_observation",
    "description": "Daily meteorological observations for Wien-Hohe Warte station, 2000-2023",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {"name": "station_id",       "type": "int",     "size": None, "d": None, "description": "Foreign key referencing the station",                   "enums": None, "sets": None, "index_length": None, "null_allowed": False, "concept_uri": None, "unit_uri": None},
        {"name": "obs_date",         "type": "date",    "size": None, "d": None, "description": "Date of the daily observation",                         "enums": None, "sets": None, "index_length": None, "null_allowed": False, "concept_uri": None, "unit_uri": None},
        {"name": "temp_mean_c",      "type": "decimal", "size": 5,    "d": 2,    "description": "Daily mean air temperature in degrees Celsius",          "enums": None, "sets": None, "index_length": None, "null_allowed": True,  "concept_uri": None, "unit_uri": None},
        {"name": "temp_max_c",       "type": "decimal", "size": 5,    "d": 2,    "description": "Daily maximum air temperature in degrees Celsius",       "enums": None, "sets": None, "index_length": None, "null_allowed": True,  "concept_uri": None, "unit_uri": None},
        {"name": "temp_min_c",       "type": "decimal", "size": 5,    "d": 2,    "description": "Daily minimum air temperature in degrees Celsius",       "enums": None, "sets": None, "index_length": None, "null_allowed": True,  "concept_uri": None, "unit_uri": None},
        {"name": "precipitation_mm", "type": "decimal", "size": 6,    "d": 2,    "description": "Daily precipitation sum in millimetres",                 "enums": None, "sets": None, "index_length": None, "null_allowed": True,  "concept_uri": None, "unit_uri": None},
        {"name": "sunshine_h",       "type": "decimal", "size": 5,    "d": 2,    "description": "Daily sunshine duration in hours",                      "enums": None, "sets": None, "index_length": None, "null_allowed": True,  "concept_uri": None, "unit_uri": None},
        {"name": "humidity_pct",     "type": "decimal", "size": 5,    "d": 2,    "description": "Daily mean relative humidity in percent",               "enums": None, "sets": None, "index_length": None, "null_allowed": True,  "concept_uri": None, "unit_uri": None},
        {"name": "visibility_m",     "type": "decimal", "size": 8,    "d": 1,    "description": "Daily mean visibility in metres",                       "enums": None, "sets": None, "index_length": None, "null_allowed": True,  "concept_uri": None, "unit_uri": None},
    ],
    "constraints": {
        "primary_key": ["station_id", "obs_date"],
        "uniques": [],
        "checks": [],
        "foreign_keys": [
            {
                "columns": ["station_id"],
                "referenced_table": "station",
                "referenced_columns": ["station_id"],
                "on_update": "cascade",
                "on_delete": "cascade"
            }
        ]
    }
}
obs_result   = post(f"/database/{database_id}/table", obs_payload)
obs_table_id = obs_result["id"]
print(f"Created table 'daily_observation' id: {obs_table_id}")

## 7 – Create citable identifier (metadata)

Attaches **DataCite Metadata Schema 4.4** metadata to the database to make it
citable with a persistent identifier (PID / DOI).

**Semantic decisions:**
- `publisher` → **GeoSphere Austria** — original data owner; we are only re-publishing
- `creators` → **GeoSphere Austria only** — they collected the data;
- `contributors` (role `DataCurator`) → team members who designed the schema and republished
- `license` → **CC0 1.0** — inherited unchanged from the source dataset
- `related_identifiers` → link back to the original GeoSphere Austria Data Hub record

In [54]:
identifier_result = post("/identifier", {
    "type": "database",
    "doi":  None,

    "titles": [
        {
            "title":    "Daily Meteorological Observations - Wien-Hohe Warte 2000-2023",
            "language": "en",
            "type":     "Subtitle"
        }
    ],

    "descriptions": [
        {
            "description": "Daily met. obs. Wien-Hohe Warte 2000-2023, GeoSphere Austria Tagesdaten v2, CC0.",
            "language":    "en",
            "type":        "Abstract"
        }
    ],

    "funders": [],

    "licenses": [
        {
            "identifier":  "CC0-1.0",
            "uri":         "https://creativecommons.org/publicdomain/zero/1.0/",
            "description": "No rights reserved. The data is in the public domain."
        }
    ],

    "publisher": "GeoSphere Austria",
    "language":  "en",

    "creators": [
        {
            "firstname":              "Sophie",
            "lastname":               "Konecny",
            "affiliation":            "TU Wien",
            "creator_name":           "Konecny, Sophie",
            "name_type":              "Personal",
            "name_identifier":        "https://orcid.org/0009-0006-5745-5729",
            "affiliation_identifier": None
        },
        {
            "firstname":              "Vivek",
            "lastname":               "Sharma",
            "affiliation":            "TU Wien",
            "creator_name":           "Sharma, Vivek",
            "name_type":              "Personal",
            "name_identifier":        "https://orcid.org/0009-0006-4879-6388",
            "affiliation_identifier": None
        },
        {
            "firstname":              "Anxhela",
            "lastname":               "Sulmina",
            "affiliation":            "TU Wien",
            "creator_name":           "Sulmina, Anxhela",
            "name_type":              "Personal",
            "name_identifier":        "https://orcid.org/0009-0008-9371-0488",
            "affiliation_identifier": None
        },
        {
            "firstname":              "Nayama",
            "lastname":               "Alam",
            "affiliation":            "TU Wien",
            "creator_name":           "Alam, Nayama",
            "name_type":              "Personal",
            "name_identifier":        None,
            "affiliation_identifier": None
        }
    ],

    "database_id":       database_id,
    "query_id":          None,
    "view_id":           None,
    "table_id":          None,
    "publication_day":   14,
    "publication_month": 5,
    "publication_year":  2026,

    "related_identifiers": [
        {
            "value":    "https://data.hub.geosphere.at/dataset/klima-v2-1d",
            "type":     "URL",
            "relation": "IsDerivedFrom"
        }
    ]
})

print("Identifier created:")
print(json.dumps(identifier_result, indent=2))

Identifier created:
{
  "id": "40294b79-1d19-4fec-b2d5-2eb523085e38",
  "links": {
    "self": "/api/v1/identifier/40294b79-1d19-4fec-b2d5-2eb523085e38",
    "data": null,
    "self_html": "/pid/40294b79-1d19-4fec-b2d5-2eb523085e38",
    "dashboard_html": "/d/cfm18q8w3ttz4c"
  },
  "type": "database",
  "titles": [
    {
      "id": "dd49965e-6421-4228-b0d6-8ff65b9b6857",
      "title": "Daily Meteorological Observations - Wien-Hohe Warte 2000-2023",
      "language": "en",
      "type": "Subtitle"
    }
  ],
  "descriptions": [
    {
      "id": "226534d1-38a3-4a6f-a928-d8665f2be372",
      "description": "Daily met. obs. Wien-Hohe Warte 2000-2023, GeoSphere Austria Tagesdaten v2, CC0.",
      "language": "en",
      "type": "Abstract"
    }
  ],
  "funders": [],
  "query": null,
  "execution": null,
  "doi": "10.82556/cxve-c208",
  "publisher": "GeoSphere Austria",
  "owner": {
    "id": null,
    "username": "datastewgroup4",
    "name": null,
    "orcid": null,
    "qualified_name"

## 8 – Summary

In [55]:
pid = identifier_result.get("doi") or identifier_result.get("id", "n/a")

print("=" * 60)
print("DBRepo objects created successfully")
print("=" * 60)
print(f"  Database id               : {database_id}")
print(f"  Table 'station'       id  : {station_table_id}")
print(f"  Table 'daily_obs'     id  : {obs_table_id}")
print(f"  Persistent identifier     : {pid}")
print()

DBRepo objects created successfully
  Database id               : a16a980a-3489-492b-adcf-74c70a07248b
  Table 'station'       id  : 9c57f7ff-99b6-454b-8d51-cff340ecf934
  Table 'daily_obs'     id  : 21e39deb-1db9-4d34-bc04-45fa5cef72a0
  Persistent identifier     : 10.82556/cxve-c208

Next steps:
  T2.2  semantic column mappings  → /api/v1/column/{id}/concept
  T2.3  unit-of-measurement maps  → /api/v1/column/{id}/unit
  T2.4  SQL VIEW definitions
  T2.5  load CSV data             → /api/v1/database/{id}/table/{id}/data
